# Inteligência Artificial (PCO118A) — Aula 6 — Laboratório: Minimax
**Prof. Carlos Costa Cedro** · Bacharelado em Ciência da Computação · PUCPR

**GABARITO DO PROFESSOR.** Todas as lacunas resolvidas e conferidas por execução.

O jogo é o da velha, e a escolha é deliberada: **Minimax puro roda até o fim sem qualquer truque**. Vocês vão obter o valor exato do jogo e a contagem exata de nós, o que produz a linha de base que será comparada com a poda alfa-beta em 16/09.

In [1]:
# ============================================================
# CÓDIGO FORNECIDO — não altere
# ============================================================
# Representação do tabuleiro: string de 9 caracteres, 'X', 'O' ou ' '.
# Índices:   0 1 2
#            3 4 5
#            6 7 8
# Strings são imutáveis em Python, o que resolve de graça a terceira
# armadilha da seção 3.5.1 do material teórico: não existe jogada a desfazer.

import time, random

LINHAS = [(0,1,2),(3,4,5),(6,7,8),   # horizontais
          (0,3,6),(1,4,7),(2,5,8),   # verticais
          (0,4,8),(2,4,6)]           # diagonais

VAZIO = ' ' * 9

def venceu(b, p):
    """True se o jogador p tem uma linha completa em b."""
    return any(all(b[i] == p for i in linha) for linha in LINHAS)

def jogadas(b):
    """Índices das casas livres. É a função AÇÕES(s) da Aula 6."""
    return [i for i in range(9) if b[i] == ' ']

def resultado(b, i, p):
    """RESULTADO(s, a): novo tabuleiro com p na casa i."""
    return b[:i] + p + b[i+1:]

def oponente(p):
    return 'O' if p == 'X' else 'X'

def mostrar(b):
    for k in (0, 3, 6):
        print(' ' + ' | '.join(b[k:k+3]))
        if k < 6: print(' ---+---+---')

print('Setup carregado.')
mostrar(resultado(resultado(VAZIO, 4, 'X'), 0, 'O'))

Setup carregado.
 O |   |  
 ---+---+---
   | X |  
 ---+---+---
   |   |  


## Lacuna 1 — TESTE-TERMINAL e UTILIDADE

Implemente os dois últimos componentes da formalização da Aula 6.

- `eh_terminal(b)` devolve `True` quando o jogo acabou.
- `utilidade(b)` devolve **sempre do ponto de vista de X**: `+1` se X venceu,
  `-1` se O venceu, `0` em empate.

> **Atenção.** A convenção de sinal é única para os dois jogadores. Inverter o sinal
> em algum lugar produz um agente que joga bem de um lado e suicidamente do outro,
> e o defeito só aparece quando se troca de lado.

In [2]:
def eh_terminal(b):
    return venceu(b, 'X') or venceu(b, 'O') or ' ' not in b

def utilidade(b):
    if venceu(b, 'X'): return  1
    if venceu(b, 'O'): return -1
    return 0

In [3]:
# Testes de sanidade (fornecidos)
assert eh_terminal('XXX OO   ') and utilidade('XXX OO   ') ==  1
assert eh_terminal('OOOXX X  ') and utilidade('OOOXX X  ') == -1
assert eh_terminal('XOXXOOOXX') and utilidade('XOXXOOOXX') ==  0
assert not eh_terminal(VAZIO) and utilidade(VAZIO) == 0
print('Lacuna 1: OK')

Lacuna 1: OK


## Lacuna 2 — VALOR-MAX e VALOR-MIN, com contador de nós

Implemente o Minimax da seção 3.5.1 do material teórico. Cada função devolve o par
`(valor, melhor_jogada)`, e `melhor_jogada` é `None` em estado terminal.

Use a variável global `NOS` para contar **todo nó visitado**, incluindo os terminais.
Incremente-a uma vez logo na entrada de cada função.

> **Atenção.** A comparação deve ser estrita (`>` em VALOR-MAX e `<` em VALOR-MIN).
> Usando `>=` o algoritmo devolve o mesmo valor mas prefere sempre a última jogada
> empatada, o que dificulta a depuração.

In [4]:
NOS = 0

def valor_max(b):
    global NOS
    NOS += 1
    if eh_terminal(b):
        return utilidade(b), None
    v, melhor = -2, None
    for i in jogadas(b):
        v2, _ = valor_min(resultado(b, i, 'X'))
        if v2 > v:
            v, melhor = v2, i
    return v, melhor

def valor_min(b):
    global NOS
    NOS += 1
    if eh_terminal(b):
        return utilidade(b), None
    v, melhor = 2, None
    for i in jogadas(b):
        v2, _ = valor_max(resultado(b, i, 'O'))
        if v2 < v:
            v, melhor = v2, i
    return v, melhor

def decisao_minimax(b, p):
    return valor_max(b) if p == 'X' else valor_min(b)

In [5]:
NOS = 0
t = time.time()
valor, jogada = decisao_minimax(VAZIO, 'X')
print(f'valor minimax do tabuleiro vazio = {valor}')
print(f'nós visitados = {NOS}')
print(f'tempo = {time.time() - t:.1f} s')

assert valor == 0,     'o jogo da velha é empate com jogo perfeito'
assert NOS   == 549946, f'esperado 549946 nós, obtido {NOS}'
print('Lacuna 2: OK')

valor minimax do tabuleiro vazio = 0
nós visitados = 549946
tempo = 3.5 s
Lacuna 2: OK


## Lacuna 3 — nós por primeira jogada

Meça o tamanho da subárvore que se abre depois de cada uma das nove primeiras
jogadas de X. Preencha `nos_por_jogada` com `{casa: nós}`.

Depois some os nove valores mais 1, referente ao nó raiz, e confira que o total
reconstrói exatamente a contagem da lacuna 2.

In [6]:
nos_por_jogada = {}
for i in range(9):
    b = resultado(VAZIO, i, 'X')
    NOS = 0
    valor_min(b)
    nos_por_jogada[i] = NOS

In [7]:
NOMES = {0:'canto', 1:'lateral', 2:'canto', 3:'lateral', 4:'centro',
         5:'lateral', 6:'canto', 7:'lateral', 8:'canto'}
for i in range(9):
    print(f'  casa {i} ({NOMES[i]:<7}): {nos_por_jogada[i]:>6} nós')

total = 1 + sum(nos_por_jogada.values())
print(f'\ntotal reconstruído = {total}')
assert total == 549946
assert nos_por_jogada[4] == 55505 and nos_por_jogada[0] == 59705 and nos_por_jogada[1] == 63905
print('Lacuna 3: OK')

  casa 0 (canto  ):  59705 nós
  casa 1 (lateral):  63905 nós
  casa 2 (canto  ):  59705 nós
  casa 3 (lateral):  63905 nós
  casa 4 (centro ):  55505 nós
  casa 5 (lateral):  63905 nós
  casa 6 (canto  ):  59705 nós
  casa 7 (lateral):  63905 nós
  casa 8 (canto  ):  59705 nós

total reconstruído = 549946
Lacuna 3: OK


### Para o relatório

As três primeiras jogadas têm o **mesmo** valor minimax, zero: todas empatam com jogo
perfeito. Mesmo assim o centro gera a menor subárvore e a lateral a maior, com 15% de
diferença. **Explique por que o custo de busca difere quando o resultado não difere.**

## Lacuna 4 — o agente joga

Monte o agente e faça-o disputar 200 partidas contra um agente aleatório, em cada lado.

A função `valor_rapido` abaixo é **fornecida**: é a mesma recursão da lacuna 2 com
memoização, para que as 400 partidas rodem em segundos. Na lacuna 5 você constrói
a sua própria.

Implemente `melhor_jogada` e `partida`.

In [8]:
# ============================================================
# CÓDIGO FORNECIDO — memoização apenas para viabilizar o torneio
# ============================================================
from functools import lru_cache

@lru_cache(maxsize=None)
def valor_rapido(b, p):
    if eh_terminal(b):
        return utilidade(b)
    vs = [valor_rapido(resultado(b, i, p), oponente(p)) for i in jogadas(b)]
    return max(vs) if p == 'X' else min(vs)

In [9]:
def melhor_jogada(b, p):
    f = max if p == 'X' else min
    return f(jogadas(b), key=lambda i: valor_rapido(resultado(b, i, p), oponente(p)))

def partida(lado_minimax, rng):
    b, p = VAZIO, 'X'
    while not eh_terminal(b):
        i = melhor_jogada(b, p) if p == lado_minimax else rng.choice(jogadas(b))
        b = resultado(b, i, p)
        p = oponente(p)
    u = utilidade(b)
    return u if lado_minimax == 'X' else -u

In [10]:
def torneio(lado, n=200, semente=1):
    rng = random.Random(semente)
    r = {'vitorias': 0, 'empates': 0, 'derrotas': 0}
    for _ in range(n):
        s = partida(lado, rng)
        r['vitorias' if s > 0 else ('derrotas' if s < 0 else 'empates')] += 1
    return r

rx = torneio('X'); ro = torneio('O')
print('minimax como X:', rx)
print('minimax como O:', ro)

assert rx['derrotas'] == 0 and ro['derrotas'] == 0
print('\nLacuna 4: OK — zero derrotas em 400 partidas')

minimax como X: {'vitorias': 198, 'empates': 2, 'derrotas': 0}
minimax como O: {'vitorias': 161, 'empates': 39, 'derrotas': 0}

Lacuna 4: OK — zero derrotas em 400 partidas


### Para o relatório

1. O agente não perdeu nenhuma das 400 partidas. **Isso é evidência de quê, exatamente?**
   Relacione com o Teorema 1 da Aula 6 e explique por que o resultado seria o mesmo com
   quatro mil partidas.
2. A divisão entre vitórias e empates **muda** se você trocar a semente; a coluna de
   derrotas **não**. Por quê?
3. Como X o agente vence quase sempre; como O, empata com muito mais frequência.
   **O agente é pior jogando de O?** Relacione com o valor do jogo, que é zero.

## Lacuna 5 — tabela de transposição

Agora acrescente memoização à **sua própria** implementação da lacuna 2, e instrumente-a
para responder duas perguntas: quantos **estados distintos** existem, e quantas
**chamadas** o algoritmo faz com a tabela ativa.

A chave da tabela é o par `(tabuleiro, jogador_da_vez)`. O tabuleiro sozinho não basta:
o mesmo desenho de peças com vezes diferentes é um estado diferente.

In [11]:
TABELA = {}
CHAMADAS = 0

def valor_memo(b, p):
    global CHAMADAS
    CHAMADAS += 1
    chave = (b, p)
    if chave in TABELA:
        return TABELA[chave]
    if eh_terminal(b):
        r = utilidade(b)
    else:
        vs = [valor_memo(resultado(b, i, p), oponente(p)) for i in jogadas(b)]
        r = max(vs) if p == 'X' else min(vs)
    TABELA[chave] = r
    return r

In [12]:
TABELA, CHAMADAS = {}, 0
v = valor_memo(VAZIO, 'X')
print(f'valor = {v}')
print(f'estados distintos armazenados = {len(TABELA)}')
print(f'chamadas com tabela ativa     = {CHAMADAS}')
print(f'sem tabela (lacuna 2)         = 549946')
print(f'redução em chamadas           = {549946 / CHAMADAS:.1f}x')

assert v == 0 and len(TABELA) == 5478 and CHAMADAS == 16168
print('\nLacuna 5: OK')

valor = 0
estados distintos armazenados = 5478
chamadas com tabela ativa     = 16168
sem tabela (lacuna 2)         = 549946
redução em chamadas           = 34.0x

Lacuna 5: OK


## Lacuna 6 — a tabela que você leva para 16/09

Monte e salve a tabela abaixo. **Ela é insumo obrigatório do laboratório da Aula 7**,
em que os mesmos números serão medidos de novo com poda alfa-beta e comparados linha a
linha. Sem esta tabela, a comparação da próxima aula não acontece.

In [13]:
import csv

linhas = [{'casa': i, 'posicao': NOMES[i], 'nos_minimax': nos_por_jogada[i]}
          for i in range(9)]
linhas.append({'casa': 'total', 'posicao': '-', 'nos_minimax': 549946})

with open('aula6_minimax.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['casa', 'posicao', 'nos_minimax'])
    w.writeheader(); w.writerows(linhas)

for l in linhas: print(l)
print('\nArquivo aula6_minimax.csv gravado. Baixe-o e traga para 16/09.')

{'casa': 0, 'posicao': 'canto', 'nos_minimax': 59705}
{'casa': 1, 'posicao': 'lateral', 'nos_minimax': 63905}
{'casa': 2, 'posicao': 'canto', 'nos_minimax': 59705}
{'casa': 3, 'posicao': 'lateral', 'nos_minimax': 63905}
{'casa': 4, 'posicao': 'centro', 'nos_minimax': 55505}
{'casa': 5, 'posicao': 'lateral', 'nos_minimax': 63905}
{'casa': 6, 'posicao': 'canto', 'nos_minimax': 59705}
{'casa': 7, 'posicao': 'lateral', 'nos_minimax': 63905}
{'casa': 8, 'posicao': 'canto', 'nos_minimax': 59705}
{'casa': 'total', 'posicao': '-', 'nos_minimax': 549946}

Arquivo aula6_minimax.csv gravado. Baixe-o e traga para 16/09.


---
## Entrega

O relatório deve conter:

- as respostas às perguntas das lacunas 3 e 4;
- **o tabuleiro inicial usado em toda medição de nós.** Contagens sem estado inicial
  declarado não são reproduzíveis e serão devolvidas;
- o arquivo `aula6_minimax.csv` da lacuna 6, obrigatório;
- uma frase indicando se a sua implementação usa estados imutáveis ou desfaz jogadas
  no retorno da recursão, conforme a terceira armadilha da seção 3.5.1 do material teórico.